# **1 - Data Loading, Initial Inspection and Auditing**

## **1.1 - Loading, and Initial Inspection**

We will begin by loading the dataset into the environment.  
The first step is to understand the overall structure of the data, including the number of rows and columns, column names, and data types. Once we have a clear picture of the dataset, we will check for any potential issues, such as missing values, duplicates, or inconsistencies.

In [4]:
import pandas as pd

In [5]:
df = pd.read_excel(r"C:\Users\Abdulaziz\.cache\kagglehub\datasets\mohammadkaiftahir\superstore-sales-dataset\versions\1\dataset.xlsx")

In [6]:
df.head()

,Row ID+O6G3A1:R6,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Profit,Returns,Payment Mode
0,4918,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,East,FUR-BO-10004709,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",73.94,1,28.2668,0,Online
1,4919,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,East,FUR-BO-10004709,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",173.94,3,38.2668,0,Online
2,4920,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,East,TEC-PH-10000455,Technology,Phones,GE 30522EE2,231.98,2,67.2742,0,Cards
3,3074,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,West,OFF-ST-10003692,Office Supplies,Storage,Recycled Steel Personal File for Hanging File ...,114.46,2,28.6150,0,Online
4,8604,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,Central,TEC-AC-10002217,Technology,Accessories,Imation Clip USB flash drive - 8 GB,30.08,2,-5.2640,0,Online


In [7]:
df.shape

(5901, 21)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Row ID+O6G3A1:R6  5901 non-null   int64         
 1   Order ID          5901 non-null   object        
 2   Order Date        5901 non-null   datetime64[ns]
 3   Ship Date         5901 non-null   datetime64[ns]
 4   Ship Mode         5901 non-null   object        
 5   Customer ID       5901 non-null   object        
 6   Customer Name     5901 non-null   object        
 7   Segment           5901 non-null   object        
 8   Country           5901 non-null   object        
 9   City              5901 non-null   object        
 10  State             5901 non-null   object        
 11  Region            5901 non-null   object        
 12  Product ID        5901 non-null   object        
 13  Category          5901 non-null   object        
 14  Sub-Category      5901 n

In [9]:
df.duplicated().sum()

0

In [10]:
df = df.drop(columns=['Row ID+O6G3A1:R6'])

We can observe that:

- The dataset contains 5,901 rows and 21 columns.  
- Column names are clear and self-explanatory.
- `Row ID+O6G3A1:R6` is a simple row identifer with no analytical value, which was dropped.
- Data types mostly make sense given the column names.  
- There are no obvious missing values.  
- No duplicate rows exist.  


## **1.2 - Data Auditing**

### **1.2.1 - Structural Audit**

Although the instial inspection gave some confidence that the dataset is structurally sound, however, missingness or issues can still be hidden, which can be masked by entries that won’t show up in a simple inspection.  

So, we will write a reusable function that audit the dataset and check for the presense and presentage of:  
- True null values (`NaN`)  
- Pseudo-null placeholders like "?", "N/A", "unknown", etc.  
- Leading or trailing whitespace  
- Special-character-only entries

In [15]:
import re

def audit_dataset(df):
    placeholders = [
        "?", "n/a", "na", "null", "none", "--", "-", "missing", "unknown", " ", "", "NaN", "undefined", "."
    ]
    
    audit_results = []

    for col in df.columns:
        col_data = df[col]
        total_rows = len(col_data)

        # Check for true NaNs
        null_count = col_data.isnull().sum()

        # Check for pseudo-nulls
        pseudo_null_mask = col_data.astype(str).str.strip().str.lower().isin([p.lower() for p in placeholders]) if col_data.dtype == 'object' else pd.Series(False, index=col_data.index)
        pseudo_nulls = col_data[pseudo_null_mask].unique().tolist()
        pseudo_null_pct = round(pseudo_null_mask.sum() / total_rows * 100, 2)

        # Check for Leading/trailing whitespace
        whitespace_count = col_data.astype(str).apply(lambda x: x != x.strip()).sum() if col_data.dtype == 'object' else 0
        whitespace_pct = round(whitespace_count / total_rows * 100, 2)

        # Check for Special character only rows
        special_only_count = col_data.astype(str).apply(lambda x: bool(re.match(r'^[^a-zA-Z0-9]+$', x)) if x.strip() else False).sum() if col_data.dtype == 'object' else 0
        special_only_pct = round(special_only_count / total_rows * 100, 2)

        audit_results.append({
            "Column": col,
            "Type": col_data.dtype,
            "True NaNs": null_count,
            "Pseudo-Nulls Found": pseudo_nulls,
            "Pseudo-Nulls %": pseudo_null_pct,
            "Whitespace Rows": whitespace_count,
            "Whitespace %": whitespace_pct,
            "Special Char Only Rows": special_only_count,
            "Special Char %": special_only_pct
        })

    return pd.DataFrame(audit_results)


audit_report = audit_dataset(df)
print(audit_report)


           Column            Type  True NaNs Pseudo-Nulls Found  \
0        Order ID          object          0                 []   
1      Order Date  datetime64[ns]          0                 []   
2       Ship Date  datetime64[ns]          0                 []   
3       Ship Mode          object          0                 []   
4     Customer ID          object          0                 []   
5   Customer Name          object          0                 []   
6         Segment          object          0                 []   
7         Country          object          0                 []   
8            City          object          0                 []   
9           State          object          0                 []   
10         Region          object          0                 []   
11     Product ID          object          0                 []   
12       Category          object          0                 []   
13   Sub-Category          object          0                 [

Based on the audit results, we can be confident that the dataset is free of:

- True missing values  
- Pseudo-null placeholders  
- Leading or trailing whitespaces  
- Special-character-only entries  



### **1.2.2 - Domain Audit**

In [18]:
integrity_check = df[df['Ship Date'] < df['Order Date']]
print(f"Logic Errors (Shipping): {len(integrity_check)}")

negative_sales = df[df['Sales'] <= 0]
print(f"Logic Errors (Negative Sales): {len(negative_sales)}")

negative_quantity = df[df['Quantity'] <= 0]
print(f"Logic Errors (Negative Quantity): {len(negative_quantity)}")

print(f"Returns values :{df["Returns"].unique()}")

Logic Errors (Shipping): 0
Logic Errors (Negative Sales): 0
Logic Errors (Negative Quantity): 0
Returns values :[0 1]


Based on the logic checks, we can confirm that the dataset is free from obvious logical errors such as:

- Shipping Date before Order Date  
- Negative sales number
- Negative quantity  
- Returns values other than 0 and 1


## **2.1 - Feature Engineering, KPI Construction, and Finalizing**

Now that we have verified the dataset’s structural integrity and confirmed the absence of missing values, logical inconsistencies, and domain violations, we can confidently proceed to feature engineering.
The goal of this step is to transform raw transactional fields into business-meaningful KPIs that capture operational performance, financial efficiency, and customer behavior.

Each KPI is engineered with a specific analytical purpose and will later support segmentation, dashboarding, and decision-making.

### **2.1.1 Operational KPI – Shipping Lag**

We begin by engineering an operational KPI, namely Shipping Lag, which measures the number of days between when an order is placed and when it is shipped.

This metric is important because it serves as a direct indicator of fulfillment efficiency. Longer shipping lags may point to logistical bottlenecks, inventory constraints, or operational inefficiencies, while shorter lags indicate smoother order processing.

Shipping Lag is calculated as the difference (in days) between the Ship Date and the Order Date.

In [24]:
df['Shipping Lag (Days)'] = (df['Ship Date'] - df['Order Date']).dt.days

### **2.1.2 Financial KPI – Profit Margin (%)**

Next, we engineer a financial KPI: Profit Margin (%).

While absolute profit values are useful, they can be misleading when comparing orders of different sizes. A small absolute loss on a large sale may be negligible, whereas the same loss on a small sale can be significant. Profit Margin normalizes profit relative to sales, allowing for fair and scalable performance comparison.

Profit Margin is calculated as profit divided by sales and expressed as a percentage.

In [27]:
df['Profit Margin %'] = ((df['Profit'] / df['Sales']) * 100).round(2)

### **2.1.3 Strategic KPI – RFM (Recency, Frequency, Monetary)**

Finally, we engineer a strategic, customer-level KPI using the RFM framework. Customers are not homogeneous, and analyzing transactions at the order level alone can hide critical behavioral differences.

The RFM model captures three complementary dimensions of customer behavior:

Recency: how recently a customer made a purchase

Frequency: how often they purchase

Monetary Value: how much revenue they generate

By aggregating transactional data at the customer level, we can distinguish high-value active customers from those who are disengaged or at risk of churn.

To streamline the process and ensure consistency, a Python RFM library is used to automate metric calculation, scoring, and segmentation. The resulting RFM scores and customer segments are then merged back into the main dataset for downstream analysis and visualization.

In [30]:
from rfm import RFM

# This will initialize RFM
rfm_model = RFM(
    df,
    customer_id='Customer ID',
    transaction_date='Order Date',
    amount='Sales'
)

# Get the results
rfm_results = rfm_model.rfm_table.copy()

# Lets make sure
print(rfm_results.columns)

Index(['Customer ID', 'recency', 'frequency', 'monetary_value', 'r', 'f', 'm',
       'rfm_score', 'segment'],
      dtype='object')


In [31]:
# Now we can safely merge
df = df.merge(
    rfm_results,
    on='Customer ID',
    how='left'
)

df[['Shipping Lag (Days)',
    'Profit Margin %',
    'recency',
    'frequency',
    'monetary_value',
    'rfm_score',
    'segment']].head()

,Shipping Lag (Days),Profit Margin %,recency,frequency,monetary_value,rfm_score,segment
0,6,38.23,101,8,1118.886,232,Lost
1,5,22,101,8,1118.886,232,Lost
2,5,29,101,8,1118.886,232,Lost
3,2,25,27,9,2668.932,444,Champions
4,5,-17.5,14,7,2478.644,534,Loyal Accounts


In [32]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Shipping Lag (Days),Profit Margin %,recency,frequency,monetary_value,r,f,m,rfm_score,segment
0,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,Maryland,...,6,38.23,101,8,1118.886,2,3,2,232,Lost
1,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,Maryland,...,5,22,101,8,1118.886,2,3,2,232,Lost
2,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,Maryland,...,5,29,101,8,1118.886,2,3,2,232,Lost
3,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,California,...,2,25,27,9,2668.932,4,4,4,444,Champions
4,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,Texas,...,5,-17.5,14,7,2478.644,5,3,4,534,Loyal Accounts


### **2.1.4 Finalizing and Preparing for Power BI**

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Order ID             5901 non-null   object        
 1   Order Date           5901 non-null   datetime64[ns]
 2   Ship Date            5901 non-null   datetime64[ns]
 3   Ship Mode            5901 non-null   object        
 4   Customer ID          5901 non-null   object        
 5   Customer Name        5901 non-null   object        
 6   Segment              5901 non-null   object        
 7   Country              5901 non-null   object        
 8   City                 5901 non-null   object        
 9   State                5901 non-null   object        
 10  Region               5901 non-null   object        
 11  Product ID           5901 non-null   object        
 12  Category             5901 non-null   object        
 13  Sub-Category         5901 non-nul

Although all rows are complete and KPIs are calculated, some numeric columns are currently stored as object, which can cause issues in aggregation, sorting, and visualization in Power BI. Let's convert them correctly.

In [53]:
numeric_cols = [
    'Sales', 'Quantity', 'Profit', 'Returns',
    'Shipping Lag (Days)', 'Profit Margin %', 'rfm_score'
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Order ID             5901 non-null   object        
 1   Order Date           5901 non-null   datetime64[ns]
 2   Ship Date            5901 non-null   datetime64[ns]
 3   Ship Mode            5901 non-null   object        
 4   Customer ID          5901 non-null   object        
 5   Customer Name        5901 non-null   object        
 6   Segment              5901 non-null   object        
 7   Country              5901 non-null   object        
 8   City                 5901 non-null   object        
 9   State                5901 non-null   object        
 10  Region               5901 non-null   object        
 11  Product ID           5901 non-null   object        
 12  Category             5901 non-null   object        
 13  Sub-Category         5901 non-nul

With all numeric conversions complete and data types verified, the dataset is now fully prepared for export and analysis in Power BI.

In [64]:
df.to_csv("final_kpi_rfm_dataset.csv", index=False)